# Streamlining EDA with a Multi-Agent System (AutoGen)

**Assignment 11.** Six AutoGen agents collaborate to take a messy dataset from raw CSV to a
finished, critic-reviewed EDA report.

| Agent | Role |
|---|---|
| `Admin` | Owns the task statement, starts the run, runs the final acceptance gate |
| `Data_Preparation_Agent` | Data cleaning and preprocessing |
| `EDA_Agent` | Statistical summarisation, insights, visualisations |
| `Report_Generator_Agent` | Writes the structured EDA report |
| `Critic_Agent` | Reviews for clarity, accuracy, actionability |
| `Executor` | Gates and runs every code block, reports results back |

**How to run in Google Colab**

1. Open the **Secrets** panel (key icon, left sidebar), add `OPENAI_API_KEY`, and turn
   *Notebook access* on for this notebook.
2. `Runtime -> Run all`. Nothing else to edit.

Expected wall time is a few minutes and a few cents on `gpt-4o-mini`.

**The security posture, up front.** This notebook lets a language model write Python and then runs
it on the host. That is the trust boundary, and it is treated as one: a deterministic AST gate
(section 5) inspects every block *before* execution and refuses anything outside a default-deny
allowlist. The model never decides whether its own code is safe to run, and it never decides who
speaks next.

---
## 1. Install

The course examples pin `autogen-agentchat==0.2.38`. **That pin no longer installs.** Every 0.2.x
release caps at `Requires-Python <3.13`, and Colab now runs Python 3.13, so pip refuses it:

```
ERROR: Ignored the following versions that require a different python version:
       0.2.38 Requires-Python <3.13,>=3.8
ERROR: No matching distribution found for autogen-agentchat==0.2.38
```

The 0.2 API did not disappear — it forked. Two lines came out of AutoGen 0.2:

| Package | What it is | API |
|---|---|---|
| `autogen-agentchat` 0.4+ | Microsoft's rewrite | new and async: `SelectorGroupChat`, `RoundRobinGroupChat`, `autogen_ext.*` |
| `ag2` 0.3 – 0.14 | the community fork continuing the 0.2 line | the classic API this course teaches |

This notebook pins **`ag2[openai]==0.9.10`**. It still imports as `autogen`, it exposes exactly the
`AssistantAgent` / `UserProxyAgent` / `GroupChat` / `GroupChatManager` API the course material uses,
and it supports Python `>=3.10,<3.14` — which covers Colab today.

Both halves of that pin are load-bearing: **AG2 1.0 is another ground-up rewrite** with a different
API again (`ag2.Agent`, `Task`, `Toolkit`), so `ag2>=1.0` will not run this notebook.

> If Colab offers to **restart the session** after the install, accept it and then
> `Runtime -> Run all` again. Nothing here depends on state from a previous run.

In [ ]:
import sys

print("python:", sys.version.split()[0])

if not ((3, 10) <= sys.version_info[:2] < (3, 14)):
    print(
        "\nWARNING: ag2 0.9.10 supports Python >=3.10,<3.14 and this runtime is outside that\n"
        "range, so the install below will fail. Check PyPI for a newer ag2 0.x release that\n"
        "still exposes the classic autogen API, and update the pin - do not move to ag2 1.x."
    )

# ag2's [openai] extra asks only for openai>=1.99.3, with no upper bound - so a future openai 4.x
# could arrive and break the client underneath a pinned ag2. Bounded here; verified on openai 3.13.
%pip install -q "ag2[openai]==0.9.10" "openai<4"

The import is a separate cell on purpose: if pip had to upgrade a package Colab already had
loaded, this is where it surfaces, and the message below tells you what to do about it.

In [ ]:
try:
    import autogen
except ModuleNotFoundError:
    raise RuntimeError(
        "autogen did not import.\n"
        "- If the install cell printed an error, read it. The usual cause is a Python version "
        "outside ag2's supported range.\n"
        "- If the install succeeded, Colab needs a restart: Runtime -> Restart session, then "
        "Run all again."
    ) from None

import openai

# Printed so the run self-documents what it actually executed against.
print("autogen (ag2):", autogen.__version__)
print("openai:       ", openai.__version__)

---
## 2. API key

One secret is required: `OPENAI_API_KEY`.

**In Colab:** Secrets panel -> add `OPENAI_API_KEY` -> toggle *Notebook access* on.

**Reviewing this outside Colab?** The loader falls back to an environment variable, then to a
`getpass` prompt, which does not echo — so the key never lands in the notebook's saved output.
Nothing is hardcoded, and only a boolean is ever printed.

In [ ]:
import os
from getpass import getpass


def load_key(name: str) -> str:
    """Read a secret from Colab Secrets, then the environment, then prompt for it."""
    try:
        from google.colab import userdata

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        # Not on Colab, or the secret is not enabled for this notebook.
        pass

    value = os.environ.get(name)
    if value:
        return value

    return getpass(f"Enter {name}: ")


os.environ["OPENAI_API_KEY"] = load_key("OPENAI_API_KEY")

MODEL = "gpt-4o-mini"

LLM_CONFIG = {
    "config_list": [{"model": MODEL, "api_key": os.environ["OPENAI_API_KEY"]}],
    "temperature": 0,      # control flow is deterministic; keep the text near-deterministic too
    "cache_seed": 42,      # re-runs hit the local cache instead of paying for the same tokens
    "timeout": 180,
}

# Boolean only - never the value.
print("OPENAI_API_KEY loaded:", bool(os.environ.get("OPENAI_API_KEY")))
print("model:", MODEL)

---
## 3. Dataset and deterministic baseline

The dataset is **synthetic** — generated here with a fixed seed so the run is reproducible and no
real or licensed data is redistributed. It is deliberately dirty, so the Data Preparation Agent has
genuine work: missing values, exact duplicate rows, case- and whitespace-variant categories,
out-of-range discounts, unparseable dates, and a handful of price outliers.

The **baseline profile** below is computed with plain pandas, before any agent exists. It is the
ground truth the acceptance gate in section 9 checks the agents' claims against — an agent cannot
invent the size of the dataset and have it pass.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

WORK_DIR = Path("eda_workspace")
WORK_DIR.mkdir(exist_ok=True)

RAW_CSV = "raw_sales_data.csv"        # relative paths only - see the code gate in section 5
CLEAN_CSV = "cleaned_sales_data.csv"
REPORT_MD = "eda_report.md"


def make_raw_dataframe(n_rows: int = 1200, seed: int = 11) -> pd.DataFrame:
    """Build a synthetic, deliberately messy retail-orders dataset."""
    rng = np.random.default_rng(seed)

    regions = rng.choice(["North", "South", "East", "West"], size=n_rows, p=[0.3, 0.25, 0.25, 0.2])
    categories = rng.choice(
        ["Electronics", "Home", "Apparel", "Grocery", "Sports"],
        size=n_rows,
        p=[0.22, 0.2, 0.24, 0.24, 0.1],
    )

    base_price = {"Electronics": 220.0, "Home": 65.0, "Apparel": 40.0, "Grocery": 12.0, "Sports": 85.0}
    unit_price = np.array([base_price[c] for c in categories]) * rng.lognormal(0.0, 0.28, n_rows)

    units_sold = rng.integers(1, 15, n_rows).astype(float)
    discount_pct = np.round(rng.choice([0.0, 0.05, 0.1, 0.15, 0.2, 0.25], size=n_rows), 2)
    customer_rating = np.clip(np.round(rng.normal(4.0, 0.8, n_rows), 1), 1.0, 5.0)

    dates = pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 365, n_rows), unit="D")

    df = pd.DataFrame(
        {
            "order_id": np.arange(100000, 100000 + n_rows),
            "order_date": dates.strftime("%Y-%m-%d"),
            "region": regions,
            "product_category": categories,
            "units_sold": units_sold,
            "unit_price": np.round(unit_price, 2),
            "discount_pct": discount_pct,
            "customer_rating": customer_rating,
            "returned": rng.choice(["Yes", "No"], size=n_rows, p=[0.12, 0.88]),
        }
    )

    # --- inject the mess -------------------------------------------------
    # Case and whitespace variants in a categorical column.
    variant_idx = rng.choice(n_rows, 150, replace=False)
    df.loc[variant_idx, "region"] = [
        v for v in rng.choice(["north ", "NORTH", " south", "East ", "west"], 150)
    ]

    # Inconsistent boolean encoding.
    bool_idx = rng.choice(n_rows, 180, replace=False)
    df.loc[bool_idx, "returned"] = df.loc[bool_idx, "returned"].map({"Yes": "Y", "No": "N"})

    # Missing values in two columns.
    df.loc[rng.choice(n_rows, 95, replace=False), "customer_rating"] = np.nan
    df.loc[rng.choice(n_rows, 60, replace=False), "units_sold"] = np.nan

    # Out-of-range discounts.
    df.loc[rng.choice(n_rows, 18, replace=False), "discount_pct"] = 1.5

    # Price outliers.
    df.loc[rng.choice(n_rows, 12, replace=False), "unit_price"] *= 40

    # Unparseable dates.
    df.loc[rng.choice(n_rows, 25, replace=False), "order_date"] = "not-a-date"

    # Exact duplicate rows.
    duplicates = df.sample(35, random_state=seed)
    df = pd.concat([df, duplicates], ignore_index=True)

    return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)


raw_df = make_raw_dataframe()
raw_df.to_csv(WORK_DIR / RAW_CSV, index=False)

BASELINE = {
    "rows": int(raw_df.shape[0]),
    "columns": int(raw_df.shape[1]),
    "column_names": list(raw_df.columns),
    "duplicate_rows": int(raw_df.duplicated().sum()),
    "missing_by_column": {c: int(raw_df[c].isna().sum()) for c in raw_df.columns},
}

print(f"wrote {WORK_DIR / RAW_CSV}")
print(f"shape: {BASELINE['rows']} rows x {BASELINE['columns']} columns")
print(f"exact duplicate rows: {BASELINE['duplicate_rows']}")
print("missing values per column:")
for col, n_missing in BASELINE["missing_by_column"].items():
    print(f"  {col:<18} {n_missing}")
raw_df.head()

---
## 4. Audit log

Every consequential action gets an entry: who acted, when, on what, and what was decided. This is
the forensic record for the run — the same reason a SIEM keeps the raw event alongside the verdict.
It records a SHA-256 prefix of each code block rather than the code itself, and never records key
material or raw model output.

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

AUDIT_LOG = []


def audit(event: str, run_id: str, **fields) -> dict:
    """Append one audit entry. Never records key material or raw model output."""
    entry = {
        "ts": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "run_id": run_id,
        "event": event,
    }
    entry.update(fields)
    AUDIT_LOG.append(entry)
    return entry


print("audit log ready")

---
## 5. The trust boundary: a deterministic code gate

An LLM writes Python; this notebook runs it. Everything downstream of that is untrusted input, so
the control has to be deterministic and it has to sit at the executor — not in a system prompt that
a model may or may not follow.

`scan_code()` parses each block with `ast` and applies **default-deny** rules:

| Rule | Why |
|---|---|
| Python blocks only, one per message | Shell blocks are an unbounded escape; `!pip` and `%magic` fail to parse and are refused |
| Import **allowlist** (pandas, numpy, matplotlib, ...) | Denylists miss things. `os`, `subprocess`, `socket`, `requests`, `shutil`, `pickle` are simply not on the list |
| Blocked call names: `eval`, `exec`, `compile`, `__import__`, `getattr`, `globals`, `locals` | Primitives for reaching capability the allowlist withheld |
| Relative paths only — no absolute path, no `..` | Confines writes to the working directory |
| No `http://`, `https://`, `ftp://`, `file://` literals | No exfiltration, no unvetted downloads, no network at all |
| 20,000 character cap per block | Bounds the blast radius of a runaway generation |

`GuardedExecutor` refuses a violating block **before** `super().execute_code_blocks()` is reached,
and returns `exit_code=1` with a fix-it message. The agent reads that like any other failure and
rewrites its code, so the guard corrects the run instead of killing it.

*Honest limitation:* this is a static gate over a scoped working directory, not a sandbox. Colab has
no Docker daemon, so `DockerCommandLineCodeExecutor` is not an option here. A production deployment
would run the executor in a container with no network namespace and treat this gate as
defence in depth, not as the boundary.

In [ ]:
import ast
import re

ALLOWED_IMPORTS = {
    "pandas", "numpy", "matplotlib", "seaborn", "json", "math", "statistics",
    "datetime", "pathlib", "textwrap", "collections", "itertools", "re", "warnings", "csv",
}

BLOCKED_CALLS = {
    "eval", "exec", "compile", "__import__", "input", "breakpoint",
    "globals", "locals", "getattr", "setattr", "delattr", "vars",
}

# Defence in depth. The import allowlist already withholds the modules these live on.
BLOCKED_ATTRS = {
    "system", "popen", "spawn", "fork", "rmtree", "unlink", "chmod",
    "check_output", "urlopen", "to_pickle", "read_pickle",
}

UNSAFE_PATH = re.compile(r"(https?://|ftp://|file://|\.\.[\\/]|^[A-Za-z]:[\\/]|^/)")

MAX_CODE_CHARS = 20_000


def scan_code(code: str, language: str) -> tuple:
    """Static, deterministic pre-execution check.

    Returns (allowed: bool, reason: str). Default-deny: anything not understood is refused.
    """
    lang = (language or "").lower()
    if lang not in {"python", "py"}:
        return False, f"language {language!r} is not executable here - python only"

    if len(code) > MAX_CODE_CHARS:
        return False, f"code block is {len(code)} chars, limit is {MAX_CODE_CHARS}"

    try:
        tree = ast.parse(code)
    except SyntaxError as exc:
        return False, f"not parseable as Python ({exc.msg}) - shell commands and ! / % magics are refused"

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                root = alias.name.split(".")[0]
                if root not in ALLOWED_IMPORTS:
                    return False, f"import {root!r} is not on the allowlist"

        elif isinstance(node, ast.ImportFrom):
            root = (node.module or "").split(".")[0]
            if root not in ALLOWED_IMPORTS:
                return False, f"import from {root!r} is not on the allowlist"

        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
            if node.func.id in BLOCKED_CALLS:
                return False, f"call to {node.func.id}() is blocked"

        elif isinstance(node, ast.Attribute):
            if node.attr in BLOCKED_ATTRS:
                return False, f"attribute .{node.attr} is blocked"

        elif isinstance(node, ast.Constant) and isinstance(node.value, str):
            if UNSAFE_PATH.search(node.value):
                return False, f"string {node.value[:60]!r} is an absolute path, a parent path, or a URL"

    return True, "ok"


print("scan_code ready - import allowlist:", len(ALLOWED_IMPORTS), "modules")

`GuardedExecutor` is where the gate is actually enforced. It refuses before
`super().execute_code_blocks()` is ever reached, and returns `exit_code=1` with a fix-it message —
which the authoring agent reads like any other failure and responds to by rewriting its code.

In [ ]:
import hashlib

from autogen.coding import LocalCommandLineCodeExecutor
from autogen.coding.base import CommandLineCodeResult   # not re-exported from autogen.coding

GATE_REFUSAL = (
    "BLOCKED BY THE CODE GATE - the block was refused and never executed.\n"
    "Reason: {reason}\n"
    "Rewrite it under the code policy: one python block, allowlisted imports only, "
    "relative paths only, no URLs, no eval/exec/getattr."
)


POSTCOND_FAILURE = (
    "\n\nDATA POST-CONDITION FAILED - the code ran, but the dataset it produced is not valid.\n"
    "{details}\n"
    "Fix the cleaning logic and resend the complete corrected block."
)


class GuardedExecutor(LocalCommandLineCodeExecutor):
    """LocalCommandLineCodeExecutor with a deterministic refusal gate in front of it.

    `post_condition`, when supplied, is a callable returning (ok, reasons). It runs after a
    successful execution; a failure is reported back to the agent as exit_code 1 so it flows
    through the existing retry path. See section 5e.
    """

    def __init__(self, *args, run_id: str = "unset", post_condition=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.run_id = run_id
        self.post_condition = post_condition

    def execute_code_blocks(self, code_blocks) -> CommandLineCodeResult:
        for block in code_blocks:
            digest = hashlib.sha256(block.code.encode("utf-8")).hexdigest()[:12]
            allowed, reason = scan_code(block.code, block.language)
            audit(
                "code_gate",
                self.run_id,
                actor="Executor",
                sha256=digest,
                language=block.language,
                decision="allow" if allowed else "block",
                reason=reason,
            )
            if not allowed:
                return CommandLineCodeResult(
                    exit_code=1,
                    output=GATE_REFUSAL.format(reason=reason),
                    code_file=None,
                )

        result = super().execute_code_blocks(code_blocks)
        audit(
            "code_exec",
            self.run_id,
            actor="Executor",
            blocks=len(code_blocks),
            exit_code=int(result.exit_code),
        )

        if result.exit_code == 0 and self.post_condition is not None:
            ok, reasons = self.post_condition()
            audit(
                "post_condition",
                self.run_id,
                actor="Executor",
                passed=ok,
                failures=len(reasons),
            )
            if not ok:
                details = "\n".join(f"- {reason}" for reason in reasons)
                return CommandLineCodeResult(
                    exit_code=1,
                    # The agent still needs to see what its own code printed.
                    output=result.output + POSTCOND_FAILURE.format(details=details),
                    code_file=result.code_file,
                )

        return result


print("code gate ready")

### 5b. The gate, proven

These run with no API key. Each should-fail case is a way an LLM-authored block could reach past
the intended boundary; each should-pass case is code the agents genuinely need to write.

In [ ]:
SHOULD_FAIL = [
    ("import os; os.system('rm -rf /')", "python", "os import"),
    ("import subprocess; subprocess.run(['ls'])", "python", "subprocess import"),
    ("import requests; requests.get('http://evil.test/x')", "python", "network import"),
    ("import pandas as pd; pd.read_csv('https://evil.test/data.csv')", "python", "URL literal"),
    ("open('/etc/passwd').read()", "python", "absolute path"),
    ("open('../../secrets.env').read()", "python", "parent-directory path"),
    ("exec('print(1)')", "python", "exec() call"),
    ("__import__('os').listdir('.')", "python", "__import__ escape"),
    ("pip install evil-package", "python", "unparseable / shell-style"),
    ("rm -rf eda_workspace", "bash", "non-python language"),
]

SHOULD_PASS = [
    ("import pandas as pd\ndf = pd.read_csv('raw_sales_data.csv')\nprint(df.shape)", "python"),
    (
        "import matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\n"
        "plt.figure()\nplt.savefig('chart.png')",
        "python",
    ),
    ("import pandas as pd\ndf = pd.DataFrame({'a': [1]})\ndf.to_csv('cleaned_sales_data.csv', index=False)", "python"),
    ("with open('eda_report.md', 'w') as fh:\n    fh.write('# Report\\n')", "python"),
]

failures = 0

print("should be BLOCKED:")
for code, lang, label in SHOULD_FAIL:
    allowed, reason = scan_code(code, lang)
    status = "LEAKED" if allowed else "blocked"
    failures += int(allowed)
    print(f"  [{status:>7}] {label:<26} {reason}")

print("\nshould be ALLOWED:")
for code, lang in SHOULD_PASS:
    allowed, reason = scan_code(code, lang)
    status = "allowed" if allowed else "FALSE POSITIVE"
    failures += int(not allowed)
    print(f"  [{status:>7}] {code.splitlines()[0][:52]:<52} {reason}")

print(f"\ngate self-test: {'PASS' if failures == 0 else f'FAIL ({failures} wrong)'}")

---
## 5c. The report contract

The required structure of the report is defined **once**, here, and then used twice: it is
interpolated into the Report Generator Agent's system message, and it is what `validate_report()`
checks before anything is rendered. One definition means the instruction and the enforcement cannot
drift apart.

`validate_report()` splits its findings:

- **Critical** — blocks acceptance: a missing section, fewer than three embedded charts, a chart
  reference pointing at a file that does not exist, any external URL, or a missing/incorrect
  dataset shape. The shape is checked against the baseline profile from section 3, which was
  computed before any agent existed, so an agent cannot invent the size of the data.
- **Advisory** — printed, does not block: backticked snake_case tokens that are not raw schema
  columns. Those are usually legitimate derived features, so flagging without blocking is the
  honest treatment — it tells a reviewer where to look.

Why block on URLs: this analysis is entirely local, so a link in the report is either a
hallucinated citation (the failure mode the course example notebook flagged) or an exfiltration
path. Neither belongs in the deliverable.

In [ ]:
import re
from pathlib import Path

REQUIRED_SECTIONS = [
    "## 1. Dataset Overview",
    "## 2. Data Preparation Summary",
    "## 3. Key Insights and Findings",
    "## 4. Visualizations",
    "## 5. Conclusions and Recommendations",
    "## 6. Critic Review Response",
]

IMAGE_REF = re.compile(r"!\[[^\]]*\]\(([^)\s]+)\)")
EXTERNAL_URL = re.compile(r"(?:https?|ftp)://")
BACKTICKED = re.compile(r"`([A-Za-z_][A-Za-z0-9_]*)`")


def validate_report(text: str, work_dir, baseline: dict) -> tuple:
    """Deterministic output validation. Returns (accepted, critical, advisory)."""
    critical, advisory = [], []
    work_dir = Path(work_dir)

    if not text.strip():
        return False, ["report is empty or missing"], []

    for section in REQUIRED_SECTIONS:
        if section not in text:
            critical.append(f"missing required section: {section}")

    images = IMAGE_REF.findall(text)
    if len(images) < 3:
        critical.append(f"report embeds {len(images)} charts; at least 3 are required")
    for image in images:
        if not (work_dir / image).exists():
            critical.append(f"embedded chart does not exist on disk: {image}")

    if EXTERNAL_URL.search(text):
        critical.append("report contains an external URL; this analysis is local-only")

    # Word-bounded so that a column count of 9 is not "found" inside a fabricated "999 rows".
    # This is a presence check on the true shape, not a semantic assertion about the sentence.
    for label, value in (("raw row count", baseline["rows"]), ("column count", baseline["columns"])):
        if not re.search(rf"\b{value}\b", text):
            critical.append(f"report does not state the true {label} ({value})")

    known = set(baseline["column_names"])
    for token in sorted(set(BACKTICKED.findall(text))):
        if "_" in token and token not in known and not token.endswith(("_csv", "_md", "_png")):
            advisory.append(
                f"`{token}` reads like a column but is not in the raw schema - "
                "confirm it is a derived feature defined in section 2"
            )

    return not critical, critical, advisory


print(f"report contract: {len(REQUIRED_SECTIONS)} required sections")

### 5d. The validator, proven

A plausible-looking report that is wrong in four ways at once. No API key needed.

In [ ]:
FABRICATED_REPORT = """## 1. Dataset Overview
The dataset contains 999 rows and 4 columns.

## 3. Key Insights and Findings
Revenue is up across the board.

![missing chart](chart_that_does_not_exist.png)

Benchmark source: https://example.com/retail-benchmarks
"""

accepted, critical, advisory = validate_report(FABRICATED_REPORT, WORK_DIR, BASELINE)

print("fabricated report accepted:", accepted, " (expected: False)")
for finding in critical:
    print("  CRITICAL:", finding)
print(f"\nvalidator self-test: {'PASS' if not accepted else 'FAIL - a bad report was accepted'}")

---
## 5e. The data contract

The report contract in 5c checks the *deliverable*. This one checks the *data* — and it exists
because of a defect observed in a real run of this notebook.

The Data Preparation Agent read `discount_pct` as a percentage when it is a fraction. Its cleaning
code was `clip(lower=0, upper=100)`, which is a no-op against values of `1.5`, and it then derived
revenue with `(1 - discount_pct / 100)`. The result: the out-of-range rows the task statement
explicitly asked it to fix survived untouched, and a mean discount of **0.15%** was applied where
**12.5%** was intended. Every revenue figure and chart downstream inherited that.

Nothing caught it. The code gate does not read data. The Critic reviews prose. The report gate
checks structure, chart existence, dataset shape and URLs — all of which passed. The run printed
`no findings` on a report whose numbers were wrong.

Two changes close it, at both ends of the contract:

1. **Say the units.** `0.15` is ambiguous on its face — 15% or 0.15%? The task statement named the
   *problem* ("out-of-range values in discount_pct") without naming the *valid range*, so the agent
   had to guess. Section 8 now declares the schema with units and a worked example. Same class of
   bug as an untyped API field.
2. **Check the result.** `validate_clean_data()` below is a deterministic post-condition on
   `cleaned_sales_data.csv`. Every rule maps to a data-quality problem the task statement names, so
   it is the machine-checkable form of that contract rather than a new opinion.

It runs inside `GuardedExecutor` right after a successful execution. A failure is returned to the
agent as `exit_code=1` with the reasons, so it flows through the retry path that already exists —
the agent fixes its cleaning and resends. No new control flow, and a bad clean can no longer reach
the EDA stage.

In [ ]:
import pandas as pd

# discount_pct is a FRACTION. 0.15 means 15% off. Anything above this is a data error.
DISCOUNT_MAX = 0.30

# Columns that must not contain nulls once cleaning is done - imputed or dropped, agent's choice.
MUST_BE_COMPLETE = ["units_sold", "unit_price", "discount_pct", "customer_rating"]


def validate_clean_data(work_dir, baseline: dict, clean_csv: str = CLEAN_CSV) -> tuple:
    """Deterministic post-condition on the cleaned dataset. Returns (ok, reasons).

    Every rule corresponds to a data-quality problem named in the task statement.
    """
    path = Path(work_dir) / clean_csv
    if not path.exists():
        return False, [f"{clean_csv} was not written"]

    try:
        df = pd.read_csv(path)
    except Exception as exc:
        return False, [f"{clean_csv} is not readable as CSV: {exc}"]

    reasons = []

    if df.empty:
        return False, [f"{clean_csv} has no rows"]

    missing_columns = [c for c in baseline["column_names"] if c not in df.columns]
    if missing_columns:
        reasons.append(f"cleaned data dropped required columns: {missing_columns}")

    if len(df) > baseline["rows"]:
        reasons.append(f"cleaned data has {len(df)} rows, more than the raw {baseline['rows']}")
    if len(df) < baseline["rows"] * 0.5:
        reasons.append(f"cleaned data has {len(df)} rows, less than half the raw {baseline['rows']}")

    if df.duplicated().sum():
        reasons.append(f"{df.duplicated().sum()} exact duplicate rows remain")

    for column in MUST_BE_COMPLETE:
        if column in df.columns and df[column].isna().any():
            reasons.append(f"{column} still has {int(df[column].isna().sum())} missing values")

    if "discount_pct" in df.columns:
        out_of_range = df[(df["discount_pct"] < 0) | (df["discount_pct"] > DISCOUNT_MAX)]
        if len(out_of_range):
            reasons.append(
                f"discount_pct has {len(out_of_range)} values outside [0, {DISCOUNT_MAX}] "
                f"(max seen: {df['discount_pct'].max()}). It is a fraction, not a percentage"
            )

    if "unit_price" in df.columns and (df["unit_price"] <= 0).any():
        reasons.append(f"unit_price has {int((df['unit_price'] <= 0).sum())} values <= 0")

    if "order_date" in df.columns and df["order_date"].astype(str).eq("not-a-date").any():
        reasons.append("order_date still contains unparsed 'not-a-date' strings")

    return not reasons, reasons


print(f"data contract ready - discount_pct must be within [0, {DISCOUNT_MAX}]")

### 5f. The data contract, proven

The first case is the exact defect seen in the real run: a cleaned file where `clip(0, 100)` left
the out-of-range discounts untouched. No API key needed.

In [ ]:
_probe = WORK_DIR / "_contract_probe.csv"


def _probe_result(frame, label):
    frame.to_csv(_probe, index=False)
    ok, reasons = validate_clean_data(WORK_DIR, BASELINE, clean_csv=_probe.name)
    print(f"  [{'accepted' if ok else 'rejected':>8}] {label}")
    for reason in reasons:
        print(f"             {reason}")
    return ok


_good = raw_df.drop_duplicates().copy()
_good["discount_pct"] = _good["discount_pct"].clip(0, DISCOUNT_MAX)
_good["order_date"] = pd.to_datetime(_good["order_date"], errors="coerce")
for _column in MUST_BE_COMPLETE:
    _good[_column] = _good[_column].fillna(_good[_column].median())

print("should be REJECTED:")
_bad_discount = _good.copy()
_bad_discount.loc[_bad_discount.index[:19], "discount_pct"] = 1.5   # the observed real defect
_failures = int(_probe_result(_bad_discount, "clip(0, 100) left discounts at 1.5"))

_with_dupes = pd.concat([_good, _good.head(20)], ignore_index=True)
_failures += int(_probe_result(_with_dupes, "duplicate rows not removed"))

_with_nulls = _good.copy()
_with_nulls.loc[_with_nulls.index[:10], "customer_rating"] = None
_failures += int(_probe_result(_with_nulls, "missing values neither imputed nor dropped"))

_unparsed = _good.copy()
_unparsed["order_date"] = _unparsed["order_date"].astype(str)
_unparsed.loc[_unparsed.index[:5], "order_date"] = "not-a-date"
_failures += int(_probe_result(_unparsed, "dates left unparsed"))

_gutted = _good.head(100)
_failures += int(_probe_result(_gutted, "most of the dataset thrown away"))

print("\nshould be ACCEPTED:")
_failures += int(not _probe_result(_good, "correctly cleaned dataset"))

_probe.unlink(missing_ok=True)
print(f"\ndata contract self-test: {'PASS' if _failures == 0 else f'FAIL ({_failures} wrong)'}")

---
## 6. The six agents

One shared **code policy** is appended to every agent that writes code, so the prompt and the gate
in section 5 say the same thing. The prompt makes compliance *likely*; the gate is what makes it
*true*.

Agent responsibilities map straight to the brief:

- **Data_Preparation_Agent** — cleaning and preprocessing
- **EDA_Agent** — statistical summarisation, insights, visualisations
- **Report_Generator_Agent** — the structured report
- **Critic_Agent** — review for clarity, accuracy, actionability (writes no code)
- **Executor** — gates, runs, and reports back on every block
- **Admin** — owns the task statement, starts the run, runs the final acceptance gate

In [ ]:
from autogen import AssistantAgent, GroupChat, GroupChatManager, UserProxyAgent

CODE_POLICY = """
CODE POLICY - enforced by a deterministic gate before anything runs. Violations are refused,
not executed, and come back to you as exitcode 1. Write code that passes it the first time.
- Exactly ONE ```python code block per message. No bash, no shell, no pip, no ! or % magics.
- Allowed imports ONLY: pandas, numpy, matplotlib, seaborn, json, math, statistics, datetime,
  pathlib, textwrap, collections, itertools, re, warnings, csv. Anything else is refused.
- Relative paths only. No absolute paths, no "..", and no URLs of any kind anywhere in the code
  or in any string literal.
- Never call eval, exec, compile, __import__, getattr, setattr, globals or locals.
- For charts: matplotlib.use("Agg") before importing pyplot, save to .png, never plt.show().
- Your code runs as a standalone script with no notebook state. Re-import and re-read what you
  need. Print everything the team needs to see - the printed output is the only thing the other
  agents can read.
- If you get exitcode 1, fix the reported problem and resend the complete corrected block.
"""

DATA_PREP_MSG = f"""You are the Data Preparation Agent. You handle data cleaning and preprocessing.

Read the raw CSV named in the task. In ONE python block:
1. Print the raw shape, dtypes, per-column missing counts, and the exact duplicate row count.
2. Clean it: parse dates (unparseable ones become NaT), normalise case and whitespace in
   categorical columns, normalise the inconsistent boolean encoding, drop exact duplicate rows,
   impute or flag missing numeric values, and clip or drop out-of-range values. Print a line
   justifying each decision - never drop rows silently.
3. Add any derived columns later analysis needs, and print how each one is computed.
4. Save the result to the cleaned CSV named in the task, with index=False.
5. Print a before/after summary: rows, columns, missing cells, duplicates.

Do the cleaning only. No charts, no analysis beyond the summary, no report.
{CODE_POLICY}"""

EDA_MSG = f"""You are the EDA Agent. You perform statistical summarisation and generate insights
and visualisations.

Read the cleaned CSV named in the task. In ONE python block:
1. Print descriptive statistics for every numeric column and value counts for every categorical one.
2. Print the numeric correlation matrix, rounded to 2 decimals.
3. Print at least three group-by comparisons a decision maker would act on - for example revenue by
   product category, return rate by region, average rating by discount band.
4. Save at least THREE charts as .png files with descriptive lowercase filenames, for example
   revenue_by_category.png. Every chart gets a title and axis labels.
5. Print the exact list of chart filenames you saved, one per line, each prefixed with "CHART: ".
6. Print 3-5 numbered insights. Every insight must cite a number you actually printed above.

Do not write the report. Do not state any number you did not compute.
{CODE_POLICY}"""

REPORT_MSG = f"""You are the Report Generator Agent. You write the final EDA report.

Read the output already printed in this conversation by the Data Preparation Agent and the EDA
Agent. In ONE python block, write the report to the report file named in the task using a
triple-quoted string, then print the whole report so the team can read it.

The report MUST contain these headings, verbatim and in this order:
{chr(10).join('  ' + section for section in REQUIRED_SECTIONS)}

Rules the automated acceptance gate enforces - break one and the report is rejected:
- State the true raw row count and column count, as digits, in section 1.
- Embed every chart the EDA Agent saved with ![description](filename.png), using the exact
  filenames it printed. At least three.
- No URLs anywhere. This analysis is local-only; there is nothing to cite.
- Every number must trace to output actually printed earlier in this conversation. Do not
  estimate, do not round beyond what was printed, do not invent benchmarks.
- Section 5 gives 3-5 concrete recommendations, each tied to a specific finding.
- In your FIRST version, section 6 contains exactly: Awaiting Critic review.
- In your REVISED version, section 6 lists each critic finding and what you changed for it, or
  why no change was warranted.
{CODE_POLICY}"""

CRITIC_MSG = """You are the Critic Agent. You review the other agents' work and give feedback that
improves clarity, accuracy and actionability.

You write NO code. Prose only.

Review the report that was just printed against three tests:
1. ACCURACY - is every number traceable to output printed earlier in this conversation? Flag
   anything that looks estimated, invented, or inconsistent with the cleaning summary.
2. CLARITY - is the required structure followed, are charts referenced where they are discussed,
   would a reader who has not seen this conversation follow it?
3. ACTIONABILITY - is each recommendation specific, tied to a finding, and something a business
   could actually do? Flag generic filler.

Output a numbered list of findings. Each one names the section, the problem, and the specific fix.
At most 6 findings, highest impact first. If something is genuinely good, say so rather than
inventing a complaint. End with the single line: FEEDBACK COMPLETE"""

ADMIN_MSG = """You are the Admin. You own the project goal and the task statement and you hold the
other agents to it. You do not write code and you do not analyse data."""

# One reference to the executor, so the run id can be stamped on it without touching internals.
# The post-condition is injected rather than hardcoded, which keeps the data contract (5e)
# defined where it belongs and the executor reusable.
CODE_EXECUTOR = GuardedExecutor(
    timeout=90,                                                    # bounded: 90s per block
    work_dir=WORK_DIR,
    post_condition=lambda: validate_clean_data(WORK_DIR, BASELINE),
)


def build_agents(llm_config, code_executor):
    """Construct the six agents named in the assignment."""
    admin = UserProxyAgent(
        name="Admin",
        system_message=ADMIN_MSG,
        human_input_mode="NEVER",          # unattended run - never blocks on a console prompt
        code_execution_config=False,       # Admin executes nothing
        max_consecutive_auto_reply=0,
    )

    data_prep_agent = AssistantAgent(
        name="Data_Preparation_Agent", system_message=DATA_PREP_MSG, llm_config=llm_config
    )
    eda_agent = AssistantAgent(name="EDA_Agent", system_message=EDA_MSG, llm_config=llm_config)
    report_agent = AssistantAgent(
        name="Report_Generator_Agent", system_message=REPORT_MSG, llm_config=llm_config
    )
    critic_agent = AssistantAgent(
        name="Critic_Agent", system_message=CRITIC_MSG, llm_config=llm_config
    )

    executor = UserProxyAgent(
        name="Executor",
        system_message="Executor. Gates and runs the code written by the other agents and reports "
                       "the exact result back. Never writes or edits code.",
        human_input_mode="NEVER",
        llm_config=False,                  # the executor makes no model calls
        code_execution_config={"executor": code_executor},
        default_auto_reply=(
            "No executable code block was found in that message. Reply with exactly one "
            "```python code block."
        ),
    )

    return {
        "admin": admin,
        "prep": data_prep_agent,
        "eda": eda_agent,
        "report": report_agent,
        "critic": critic_agent,
        "executor": executor,
    }


AGENTS = build_agents(LLM_CONFIG, CODE_EXECUTOR)

for key, agent in AGENTS.items():
    print(f"{key:<9} -> {agent.name}")

---
## 7. Orchestration: the pipeline is a state machine, not a vote

AutoGen's default `speaker_selection_method="auto"` asks an LLM who should talk next. That costs a
call per turn, it is not reproducible, and it puts a probabilistic model in charge of control flow.

`select_speaker()` is a plain function instead — the same shape as a SOAR playbook: fixed stages,
one explicit feedback loop, a bounded retry budget.

```
Admin -> Prep   -> Executor --+
         EDA    -> Executor --+--> exitcode != 0 : back to the same author (max 2 fixes)
         Report -> Executor --+
                                 \
                                  -> Critic -> Report -> Executor -> done
```

The critic hop is a hard-coded transition, which is what turns "iteratively refine the analysis
based on feedback" into a guarantee rather than a hope. Termination is owned by this function too:
no agent can end the run by emitting a magic word.

In [ ]:
MAX_FIXES = 2      # retries per stage before the run is abandoned
MAX_ROUND = 30     # hard ceiling on total turns


class PipelineState:
    """Mutable state for one pipeline run. Deliberately kept outside the agents."""

    def __init__(self, agents):
        self.agents = agents
        self.run_id = "unset"
        self.reset()

    def reset(self):
        self.stage = "prep"
        self.coder = self.agents["prep"]
        self.fixes = 0
        self.revised = False
        self.aborted = None


STATE = PipelineState(AGENTS)


def select_speaker(last_speaker, groupchat):
    """Deterministic speaker selection. Returning None ends the group chat cleanly."""
    agents = STATE.agents
    content = (groupchat.messages[-1].get("content") or "") if groupchat.messages else ""

    def go(next_agent, note):
        audit(
            "transition",
            STATE.run_id,
            **{"from": last_speaker.name, "to": next_agent.name, "stage": STATE.stage, "note": note},
        )
        return next_agent

    if last_speaker is agents["admin"]:
        STATE.reset()
        return go(agents["prep"], "run start")

    # Anyone who writes code hands off to the executor. Always.
    if last_speaker in (agents["prep"], agents["eda"], agents["report"]):
        return go(agents["executor"], f"execute {STATE.stage} code")

    # The critic sends the report back for exactly one revision.
    if last_speaker is agents["critic"]:
        STATE.revised = True
        STATE.stage = "report"
        STATE.coder = agents["report"]
        STATE.fixes = 0
        return go(agents["report"], "revise with critic feedback")

    if last_speaker is agents["executor"]:
        if "exitcode: 0" not in content.lower():
            STATE.fixes += 1
            if STATE.fixes > MAX_FIXES:
                STATE.aborted = f"stage '{STATE.stage}' failed {STATE.fixes} times"
                audit("abort", STATE.run_id, stage=STATE.stage, reason=STATE.aborted)
                return None
            return go(STATE.coder, f"retry {STATE.fixes}/{MAX_FIXES}")

        STATE.fixes = 0
        if STATE.stage == "prep":
            STATE.stage, STATE.coder = "eda", agents["eda"]
            return go(agents["eda"], "cleaning accepted")
        if STATE.stage == "eda":
            STATE.stage, STATE.coder = "report", agents["report"]
            return go(agents["report"], "analysis accepted")
        if STATE.stage == "report":
            if not STATE.revised:
                STATE.stage = "critique"
                return go(agents["critic"], "first report ready for review")
            audit("pipeline_complete", STATE.run_id, stage="done")
            return None

    return None


groupchat = GroupChat(
    agents=[AGENTS[key] for key in ("admin", "prep", "eda", "report", "critic", "executor")],
    messages=[],
    max_round=MAX_ROUND,
    speaker_selection_method=select_speaker,
)

manager = GroupChatManager(
    groupchat=groupchat,
    llm_config=LLM_CONFIG,
    # Control flow belongs to select_speaker. An agent must not be able to end the run by
    # printing the word TERMINATE, which is the AutoGen 0.2 default.
    is_termination_msg=lambda message: False,
)

print(f"group chat ready: {len(groupchat.agents)} agents, max_round={MAX_ROUND}, "
      "speaker selection = deterministic state machine")

---
## 8. Run the pipeline

The task statement is Admin's contract with the team: the files, the schema, the stage order, and
the definition of done. It is built from the real column names and the real row count, so nothing
about the dataset is guessed.

Expect a long transcript below — that is the point. You can watch each handoff, each execution
result, and the critic's findings arriving before the rewrite.

In [ ]:
TASK = f"""EDA project: {RAW_CSV}, a synthetic retail orders extract.

The working directory contains {RAW_CSV}: {BASELINE['rows']} rows, {BASELINE['columns']} columns.

SCHEMA - read the units carefully, they are not guessable from the values alone:
  order_id          integer identifier, unique per original order
  order_date        date as YYYY-MM-DD; some rows hold unparseable strings
  region            categorical; case and whitespace variants present
  product_category  categorical
  units_sold        integer count; some values missing
  unit_price        currency per unit, strictly > 0; a few extreme outliers
  discount_pct      FRACTION, not a percentage. Valid range [0.0, 0.25].
                    0.15 means 15% off, so price after discount is unit_price * (1 - 0.15).
                    Any value above {DISCOUNT_MAX} is invalid data - correct or drop those rows.
  customer_rating   float 1.0 to 5.0; some values missing
  returned          Yes / No / Y / N; normalise to a single encoding

Known data quality problems to handle: missing values, exact duplicate rows, case and whitespace
variants in region, inconsistent encoding in returned, out-of-range values in discount_pct,
extreme outliers in unit_price, and unparseable strings in order_date.

The cleaned file is checked automatically after your code runs. It must have: every column above
still present, no exact duplicate rows, no missing values in units_sold / unit_price /
discount_pct / customer_rating, discount_pct within [0, {DISCOUNT_MAX}], unit_price > 0, no
unparsed date strings, and between half and all of the original {BASELINE['rows']} rows. If it
fails you get exitcode 1 with the reasons, and you fix it and resend.

Pipeline, in order:
1. Data_Preparation_Agent cleans {RAW_CSV} and writes {CLEAN_CSV}.
2. EDA_Agent analyses {CLEAN_CSV}, saves at least 3 .png charts, and prints its insights.
3. Report_Generator_Agent writes {REPORT_MD}.
4. Critic_Agent reviews it.
5. Report_Generator_Agent rewrites {REPORT_MD}, answering the critic in section 6.

Definition of done: {REPORT_MD} exists, has all six required sections, states the true raw row and
column counts, embeds at least 3 charts that exist on disk, contains no URLs, and every number in
it traces to output printed in this conversation.

Data_Preparation_Agent, start."""


def run_eda_pipeline() -> str:
    """Run one full pipeline. Returns the run id."""
    run_id = uuid4().hex[:8]

    STATE.run_id = run_id
    STATE.reset()
    CODE_EXECUTOR.run_id = run_id

    groupchat.messages.clear()
    manager.reset()
    for agent in groupchat.agents:
        agent.reset()

    audit("run_start", run_id, actor="Admin", model=MODEL, dataset=RAW_CSV,
          rows=BASELINE["rows"], columns=BASELINE["columns"])

    AGENTS["admin"].initiate_chat(manager, message=TASK, clear_history=True)

    audit("run_end", run_id, actor="Admin", turns=len(groupchat.messages),
          aborted=STATE.aborted or "no")
    return run_id


RUN_ID = run_eda_pipeline()

print(f"\n{'=' * 70}")
print(f"run {RUN_ID} finished - turns: {len(groupchat.messages)}, aborted: {STATE.aborted or 'no'}")

---
## 9. Acceptance gate

Admin's last job. The report is untrusted model output until `validate_report()` — defined and
self-tested back in section 5c, before any agent existed — has passed it.

In [ ]:
report_path = WORK_DIR / REPORT_MD
report_text = report_path.read_text(encoding="utf-8") if report_path.exists() else ""

accepted, critical, advisory = validate_report(report_text, WORK_DIR, BASELINE)

audit("report_validation", RUN_ID, actor="Admin", accepted=accepted,
      critical=len(critical), advisory=len(advisory))

print(f"report:   {report_path}")
print(f"accepted: {accepted}\n")

for finding in critical:
    print("  CRITICAL:", finding)
for finding in advisory:
    print("  advisory:", finding)
if not critical and not advisory:
    print("  no findings")

---
## 10. The final report

Rendered only because the gate above accepted it.

In [ ]:
from IPython.display import Image, Markdown, display

if accepted:
    display(Markdown(report_text))
else:
    print("REJECTED by the acceptance gate - not rendering this as a finished deliverable.")
    print("Raw text follows, for debugging:\n")
    print(report_text[:4000])

### Charts generated by the EDA Agent

In [ ]:
charts = sorted(WORK_DIR.glob("*.png"))

if charts:
    for chart in charts:
        print(chart.name)
        display(Image(filename=str(chart)))
else:
    print("no charts were produced - check the transcript above")

---
## 11. Audit log

Every gate decision, execution, state transition and validation outcome for this run. This is what
you would hand to someone asking "what did the agents actually do on this machine?"

In [ ]:
import json

print(f"{len(AUDIT_LOG)} audit entries\n")
for entry in AUDIT_LOG:
    print(json.dumps(entry))

decisions = [entry.get("decision") for entry in AUDIT_LOG if entry["event"] == "code_gate"]
print(f"\ncode gate: {decisions.count('allow')} allowed, {decisions.count('block')} blocked")

---
## 12. Limitations

Stated plainly, because a reviewer should not have to find these themselves.

- **The dataset is synthetic.** Generated in section 3 with a fixed seed. The numbers in the report
  describe that generated data and nothing else. No real retail data, no benchmark claims.
- **The code gate is not a sandbox.** It is a static AST check over a scoped working directory. It
  stops the failure modes it enumerates; it is not a proof of containment. Colab offers no Docker
  daemon, so `DockerCommandLineCodeExecutor` was not available. A production deployment would run
  the executor in a container with no network namespace and keep this gate as defence in depth.
- **The report is model-written prose.** The acceptance gate checks structure, chart existence, the
  true dataset shape, and the absence of URLs. It does not verify every arithmetic claim against the
  data — an incorrect but well-formed number can still pass. Section 3's baseline profile is the
  ground truth to check against by hand.
- **One revision cycle.** The critic speaks once and the report is rewritten once. That is enough to
  demonstrate feedback-driven refinement; it is not a convergence guarantee.
- **Bounded retries.** Two fixes per stage. A stage that fails three times aborts the run rather
  than burning tokens, and `STATE.aborted` records why.
- **Reproducibility.** `temperature=0` and `cache_seed=42` make re-runs cheap and close to
  deterministic, but LLM output is never byte-identical across model versions.